# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.PRODUITS = Set()
model.MACHINES = Set()
model.PRODUCTION = Set(dimen=2, initialize=lambda m: [(i0,i1) for i0 in m.MACHINES for i1 in m.PRODUITS])

## 🔹 Parameters

In [ ]:
model.Profit = Param(model.PRODUITS, within=NonNegativeReals)
model.Demande = Param(model.PRODUITS, within=NonNegativeReals)
model.CAP = Param(model.MACHINES, within=NonNegativeReals)
model.HRPROD = Param(model.MACHINES, model.PRODUITS, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.PRODUITS, domain=NonNegativeIntegers)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Omega_data.dat')

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for p in model.PRODUITS:
    model.c_for_0.add(model.X[p] <= model.Demande[p])
model.c_for_1 = ConstraintList()
for m in model.MACHINES:
    model.c_for_1.add(sum(model.HRPROD[m, p] * model.X[p] for p in model.PRODUITS) <= model.CAP[m])
# @BIN/@GIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Profit[p] * model.X[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')